# Export trained model to GGUF (CPU format)

Free cloud tiers have no GPU, so the model is converted to GGUF (4-bit, ~2-3 GB) and served on CPU.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Have `trained_aems_model.zip` ready to upload.
Run the cells in order. The last cell checks the converted model before you build the Docker image.

In [ ]:
# 1. Install (same pinned approach that worked for training/inference)
!pip install -q "transformers<5.0.0" bitsandbytes accelerate unsloth

In [ ]:
# 2. Upload and unzip your trained model
from google.colab import files
files.upload()   # choose trained_aems_model.zip
!unzip -q -o trained_aems_model.zip
!ls trained_aems_model

In [ ]:
# 3. Load model + adapter
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="trained_aems_model",
    max_seq_length=3072,
    dtype=torch.float16,
    load_in_4bit=True,
)
print("loaded")

In [ ]:
# 4. Merge adapter into the model and export 4-bit GGUF (takes ~10 minutes)
model.save_pretrained_gguf("aems_gguf", tokenizer, quantization_method="q4_k_m")

In [ ]:
# 5. Find the Q4_K_M file and copy it to Google Drive as aems-q4.gguf
import glob, os, shutil
from google.colab import drive
drive.mount('/content/drive')

found = [p for p in glob.glob("/content/**/*.gguf", recursive=True)
         if "/content/drive" not in p and "q4_k_m" in p.lower()]
print("candidates:", found)
assert found, "No Q4_K_M .gguf found - run: !find /content -name '*.gguf'"
src = found[0]
dst = "/content/drive/MyDrive/aems-q4.gguf"
shutil.copy(src, dst)
print(f"copied {src} -> {dst}  ({os.path.getsize(dst)/1e9:.2f} GB)")

In [ ]:
# 6. Check the GGUF behaves like the original model BEFORE building Docker.
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [ ]:
import json, time
from llama_cpp import Llama

llm = Llama(model_path="/content/drive/MyDrive/aems-q4.gguf", n_ctx=3072, n_threads=2, verbose=False)

r = {
    "use_case_name": "4.3 Manage Holiday Calendar",
    "operation": "Add",
    "description": "This use case allows an HR Administrator to add a new holiday to the organisation's yearly holiday calendar so that it is reflected in attendance and leave calculations.",
    "actors": "HR Administrator",
    "basic_flow": "1. The HR Administrator logs in to the HRMS portal.\n2. The HR Administrator navigates to Admin > Holiday Calendar.\n3. The HR Administrator clicks 'Add Holiday'.\n4. The system displays a form with the fields: Holiday Name (V100, mandatory), Holiday Date (date, mandatory), Holiday Type (dropdown: National/State/Optional, mandatory), Applicable Office (dropdown, mandatory), Description (V250, optional), Status (Active/Inactive).\n5. The HR Administrator fills in the details and clicks 'Save'.\n6. The system validates the mandatory fields and checks that no holiday already exists on the same date for the selected office.\n7. The system saves the record and displays the message 'Holiday added successfully'.\n8. The system refreshes the holiday list.",
    "alternate_flow": "A1. Mandatory field missing: the system highlights the field and shows 'Please fill all mandatory fields'.\nA2. Duplicate holiday: the system shows 'A holiday already exists on this date for the selected office'.\nA3. The HR Administrator clicks 'Cancel': the form closes without saving."
}

user = (
    f"Target Operation: {r['operation']}\n\n"
    f"Use Case Name: {r['use_case_name']}\n"
    f"Description: {r['description']}\n"
    f"Actors: {r['actors']}\n"
    f"Basic Flow Of Events:\n{r['basic_flow']}\n"
    f"Alternate Flow of Events:\n{r['alternate_flow']}"
)
prompt = f"<|system|>\nExtract FPA metrics.<|end|>\n<|user|>\n{user}<|end|>\n<|assistant|>\n"
tokens = llm.tokenize(prompt.encode("utf-8"), add_bos=True, special=True)

dec = json.JSONDecoder()
text, obj, start = "", None, time.time()
for chunk in llm(tokens, max_tokens=600, temperature=0.0, stream=True, stop=["<|end|>"]):
    text += chunk["choices"][0]["text"]
    if text.rstrip().endswith("}"):
        try:
            obj, _ = dec.raw_decode(text.lstrip())
            break
        except json.JSONDecodeError:
            pass
print(f"{time.time()-start:.0f} s on this CPU")
print(json.dumps(obj, indent=2) if obj else text)
# EXPECTED: valid JSON with keys like Complexity, OS Operation, IlfDataElements, EiDataElements, EqDataElements.
# If you see garbage instead of JSON, STOP and send me the output before building the Docker image.